In [ ]:
import pandas as pd
from datetime import datetime
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.offline as pyo
from mpl_toolkits.basemap import Basemap
import matplotlib.pyplot as plt
import numpy as np

import utils
import tracker

# Import Telemetry Dataframe

In [ ]:
telem_filename = "telem.csv"
telem_df = pd.read_csv(telem_filename)
print(f"Read {len(telem_df)} rows")

In [ ]:
telem_df_pruned = telem_df.drop_duplicates(subset=['time'])
print(f"Removed {len(telem_df) - len(telem_df_pruned)} duplicate spots")

# Parse Data

## Extract all Telemetry

In [ ]:
loc_df = pd.DataFrame()
alt_df = pd.DataFrame()
adc_df = pd.DataFrame()

for row in telem_df_pruned.itertuples():
    telem_min = datetime.strptime(row.time, "%Y-%m-%d %H:%M:%S").minute

    wspr_dict = {
        "time": row.time,
        "wspr_sign": row.tx_sign,
        "wspr_loc": row.tx_loc,
        "wspr_pwr": row.power,
        "frequency": row.frequency
    }
    
    # Grab location + sat count telem
    if (telem_min % 10 == 2):
        loc_dict = utils.decode_w6nxp_subsquare_telem(row.tx_sign, row.tx_loc, row.power)
        loc_dict |= wspr_dict
        loc_df = pd.concat([loc_df, pd.DataFrame([loc_dict])], ignore_index=True)

    # Grab altitude + speed telem
    elif (telem_min % 10 == 4):
        alt_dict = utils.decode_w6nxp_alt_telem(row.tx_sign, row.tx_loc, row.power)
        alt_dict |= wspr_dict
        alt_df = pd.concat([alt_df, pd.DataFrame([alt_dict])], ignore_index=True)

    # Grab ADC and temperature telem
    elif (telem_min % 10 == 6):
        adc_dict = utils.decode_w6nxp_adc_telem(row.tx_sign, row.tx_loc, row.power)
        adc_dict |= wspr_dict
        adc_df = pd.concat([adc_df, pd.DataFrame([adc_dict])], ignore_index=True)

## Plot Map

In [ ]:
loc_df_filtered = loc_df.loc[loc_df.wspr_loc != "JJ00"]
loc_df_filtered

### Plot track on world map

In [ ]:
plt.figure(figsize=(16, 9))

m = Basemap(projection='cyl',llcrnrlat=-90,urcrnrlat=90,\
            llcrnrlon=-180,urcrnrlon=180,resolution='c')

m.drawcoastlines()
m.drawcountries()
m.drawparallels(np.arange(-90.,91.,10.),labels=[True, False, False, True])
m.drawmeridians(np.arange(-180.,181.,20.),labels=[True, False, False, True])

#m.etopo()
m.shadedrelief()
#m.fillcontinents(color='coral',lake_color='aqua')
#m.drawmapboundary(fill_color='aqua') 

m.plot(x=loc_df_filtered.long, y=loc_df_filtered.lat, latlon=True, color='r', marker='o', markersize=4)

plt.title("Balloon Track - World Map")
plt.show()

### Plot track on map of the US

In [ ]:
plt.figure(figsize=(16, 9))

ll_lat = 20.0
ur_lat = 55.0
ll_lon = -130.0
ur_lon = -60.0

par_range = np.arange(ll_lat,ur_lat+1,5.)
mer_range = np.arange(ll_lon,ur_lon+1,5.)

m = Basemap(projection='merc',llcrnrlat=ll_lat,urcrnrlat=ur_lat,\
            llcrnrlon=ll_lon,urcrnrlon=ur_lon,lat_ts=20,resolution='l')

m.drawparallels(par_range,labels=[True, False, False, True])
m.drawmeridians(mer_range,labels=[True, False, False, True])
m.drawcoastlines()
m.drawcountries()
m.drawstates()

#m.etopo()
m.shadedrelief()
#m.fillcontinents(color='coral',lake_color='aqua')
#m.drawmapboundary(fill_color='aqua') 

m.plot(x=loc_df_filtered.long, y=loc_df_filtered.lat, latlon=True, color='r', marker='o', markersize=4)

plt.title("Balloon Track - US Map")
plt.show()

## Plot Scalar Telemetry

### Sat Count

In [ ]:
fig = go.Figure(data=go.Scatter(x=loc_df.time, y=loc_df.satellites, mode='lines+markers'))

fig.update_layout(
    xaxis_title="Time", yaxis_title="GPS Satellite Count", title="Balloon Satellite Count vs Time",
    width=1000, height=600)
fig.show()

### Barometric Pressure and Altitude

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Scatter(x=alt_df.time, y=alt_df.pressure, name="Pressure", mode='lines+markers'))
fig.add_trace(go.Scatter(x=alt_df.time, y=alt_df.altitude, name="Altitude"), secondary_y=True)

fig.update_layout(xaxis_title="Time", yaxis_title="Barometric Pressure (mBar)", title="Balloon Barometric Pressure and Altitude vs Time",
                 width=1000, height=600)
fig.update_yaxes(title_text="Altitude (m)", secondary_y=True)
fig.show()

### Speed

In [ ]:
fig = go.Figure(data=go.Scatter(x=alt_df.time, y=alt_df.speed, mode='lines+markers'))

fig.update_layout(
    xaxis_title="Time", yaxis_title="Speed (knots)", title="Balloon Speed vs Time",
    width=1000, height=600
)
fig.show()

### Temperature

In [ ]:
fig = go.Figure(data=go.Scatter(x=adc_df.time, y=adc_df.temp, mode='lines+markers'))

fig.update_layout(
    xaxis_title="Time", yaxis_title="Temperature (C)", title="Balloon Temperature vs Time",
    width=1000, height=600
)
fig.show()

In [ ]:
# Plot temp vs TX frequency to get a sense of tempco
fig = go.Figure(data=go.Scatter(x=adc_df.temp, y=adc_df.frequency - 14.097e6, mode='markers'))

fig.update_layout(
    xaxis_title="Temperature (C)", yaxis_title="TX Freq (Hz, offset from 14.097 MHz)", title="Balloon TX Frequency vs Temperature",
    width=1000, height=600
)
fig.show()

### Voltage Rails

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": False}]])

fig.add_trace(go.Scatter(x=adc_df.time, y=adc_df.v_solar, name="V_SOLAR", mode='lines+markers'))
fig.add_trace(go.Scatter(x=adc_df.time, y=adc_df.v_in, name="V_IN"))

fig.update_layout(xaxis_title="Time", yaxis_title="Voltage (V)", title="Balloon Rail Voltage vs Time",
                 width=1000, height=600)
fig.show()

### Light Sensors

In [ ]:
fig = make_subplots(specs=[[{"secondary_y": False}]])

fig.add_trace(go.Scatter(x=adc_df.time, y=adc_df.l_front, name="Front", mode='lines+markers'))
fig.add_trace(go.Scatter(x=adc_df.time, y=adc_df.l_back, name="Back"))

fig.update_layout(xaxis_title="Time", yaxis_title="Sensor Magnitude", title="Light Sensor Intensity vs Time",
                 width=1000, height=600)
fig.show()